# Lesson 1: Simple ReAct Agent from Scratch

* Importing required packages
* Loading environment variables

In [1]:
import re
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage


_ = load_dotenv()

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.05)

Sanity check if the we are able to access the model

In [4]:
query = "What is the population on the earth"
out = llm.invoke(query)
out.content

'As of my last update in October 2023, the estimated global population was around 8 billion people. For the most current population figures, I recommend checking reliable sources such as the United Nations or the World Bank, as population numbers are constantly changing.'

Writing custom tools as per our requirements

In [5]:
@tool
def calculate(what: str):
    """
    Evaluates a mathematical expression and returns the result.

    Args:
        expression: A valid Python math expression as a string.
                    Example: "2 ** 10 + 500 / 4"
    """
    return eval(what)


In [6]:
@tool
def average_dog_weight(name: str):
    """
    Returns the average weight of a dog when given the breed
    Args:
        name: The breed of the dog
        Example: "Scottish Terrier"
    :param name:
    :return:
    """
    if name in "Scottish Terrier":
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")



Please modify the prompt as per you requirements

In [7]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Question: How much does a Bulldog weigh?
Observation: A Bulldog weights 51 lbs

You then output:
Answer: A bulldog weights 51 lbs
""".strip()

* Preparing a tool map to
* Use when the name comes the output tool calls

In [8]:
tools = [calculate, average_dog_weight]
tool_map: dict = {t.name: t for t in tools}
tool_map

{'calculate': StructuredTool(name='calculate', description='Evaluates a mathematical expression and returns the result.\n\nArgs:\n    expression: A valid Python math expression as a string.\n                Example: "2 ** 10 + 500 / 4"', args_schema=<class 'langchain_core.utils.pydantic.calculate'>, func=<function calculate at 0x12465cea0>),
 'average_dog_weight': StructuredTool(name='average_dog_weight', description='Returns the average weight of a dog when given the breed\nArgs:\n    name: The breed of the dog\n    Example: "Scottish Terrier"\n:param name:\n:return:', args_schema=<class 'langchain_core.utils.pydantic.average_dog_weight'>, func=<function average_dog_weight at 0x12465dda0>)}

* Binding to tool to llms

In [9]:
llm_with_tools = llm.bind_tools(tools)

In [10]:
out = llm_with_tools.invoke("How much does a toy poodle weigh?")

In [11]:
out.tool_calls

[{'name': 'average_dog_weight',
  'args': {'name': 'Toy Poodle'},
  'id': 'call_x8WuUPBw0s8yOKG6FY097zQ2',
  'type': 'tool_call'}]

In [11]:
while out.tool_calls:
    for tc in out.tool_calls:
        tool_name = tc["name"]
        tool_args = tc["args"]
        tool_id   = tc["id"]

        print(f"\n  [Tool Called]  {tool_name}")
        print(f"  [Args]         {tool_args}")

        if tool_name not in tool_map:
            tool_output = f"Error: tool '{tool_name}' not found."
        else:
            tool_output = tool_map[tool_name].invoke(tool_args)

        print(f"  [Tool Result]  {tool_output}")
    break


  [Tool Called]  average_dog_weight
  [Args]         {'name': 'Toy Poodle'}
  [Tool Result]  a toy poodles average weight is 7 lbs


In [12]:
def run_agent(user_query: str, prompt=prompt) -> str:
    """
    Sends the user query to the LLM, executes any requested tool calls,
    feeds results back, and returns the final natural-language answer.
    """
    print(f"\n{'='*60}")
    print(f"  USER: {user_query}")
    print(f"{'='*60}")
    user_query = prompt +"\n\n User query"+user_query
    messages = [HumanMessage(content=user_query)]

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    while response.tool_calls:
        for tc in response.tool_calls:
            tool_name = tc["name"]
            tool_args = tc["args"]
            tool_id   = tc["id"]

            print(f"\n  [Tool Called]  {tool_name}")
            print(f"  [Args]         {tool_args}")

            if tool_name not in tool_map:
                tool_output = f"Error: tool '{tool_name}' not found."
            else:
                tool_output = tool_map[tool_name].invoke(tool_args)

            print(f"  [Tool Result]  {tool_output}")

            messages.append(
                ToolMessage(content=str(tool_output), tool_call_id=tool_id)
            )

        response = llm_with_tools.invoke(messages)
        messages.append(response)

    final_answer = response.content
    print(f"\n  AGENT: {final_answer}\n")
    return final_answer

In [13]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""

In [14]:
out = run_agent(question)


  USER: I have 2 dogs, a border collie and a scottish terrier. What is their combined weight

  [Tool Called]  average_dog_weight
  [Args]         {'name': 'Border Collie'}
  [Tool Result]  a Border Collies average weight is 37 lbs

  [Tool Called]  average_dog_weight
  [Args]         {'name': 'Scottish Terrier'}
  [Tool Result]  Scottish Terriers average 20 lbs

  AGENT: Answer: The combined weight of a Border Collie (37 lbs) and a Scottish Terrier (20 lbs) is 57 lbs.

